# 04 — SERCOP: consolidación y limpieza

**Fuente:** API pública de Compras Públicas del Ecuador (datosabiertos.compraspublicas.gob.ec)  
**Búsqueda:** `insulina`, años 2015–2024  
**Script de extracción:** `data/raw/SERCOP/fetch_sercop.py`  

**Objetivo:** Consolidar los 10 CSVs anuales en un único dataset limpio, agregar por provincia y año, y exportar a `processed/` para los cruces posteriores.

In [1]:
import pandas as pd
import glob
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_DIR = Path('../raw/SERCOP')
PROCESSED_DIR = Path('../processed')
PROCESSED_DIR.mkdir(exist_ok=True)

## 1. Carga y unión de los 10 CSVs

In [2]:
archivos = sorted(RAW_DIR.glob('insulina_*.csv'))
print(f'Archivos encontrados: {len(archivos)}')
for f in archivos:
    print(f'  {f.name}')

df = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)
print(f'\nShape inicial: {df.shape}')

Archivos encontrados: 10
  insulina_2015.csv
  insulina_2016.csv
  insulina_2017.csv
  insulina_2018.csv
  insulina_2019.csv
  insulina_2020.csv
  insulina_2021.csv
  insulina_2022.csv
  insulina_2023.csv
  insulina_2024.csv

Shape inicial: (7226, 15)


## 2. Inspección inicial

In [3]:
df.dtypes

id                 int64
ocid              object
year               int64
month              int64
method            object
internal_type     object
region            object
locality          object
buyer             object
suppliers         object
amount           float64
budget           float64
date              object
title             object
description       object
dtype: object

In [4]:
# Valores nulos por columna
nulos = df.isnull().sum()
print('Nulos por columna:')
print(nulos[nulos > 0])

Nulos por columna:
method            25
internal_type     25
region            20
locality          20
suppliers        122
amount            30
budget           202
title             25
description       25
dtype: int64


In [5]:
# Distribución por año
df.groupby('year').size().rename('contratos')

year
2015    1120
2016    1020
2017     726
2018     664
2019    1001
2020     837
2021     948
2022     561
2023     201
2024     148
Name: contratos, dtype: int64

## 3. Limpieza

### 3.1 Tipos de datos

In [6]:
# amount y budget a float
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')

# date a datetime
df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')

# region y locality a string limpio
df['region'] = df['region'].str.strip().str.upper()
df['locality'] = df['locality'].str.strip().str.upper()

print('Tipos después de limpieza:')
print(df[['amount', 'budget', 'date', 'region']].dtypes)

Tipos después de limpieza:
amount                float64
budget                float64
date      datetime64[ns, UTC]
region                 object
dtype: object


### 3.2 Revisión de nulos post-limpieza

In [7]:
# Registros con amount nulo o cero
sin_monto = df[df['amount'].isna() | (df['amount'] == 0)]
print(f'Registros sin monto válido: {len(sin_monto)}')
if len(sin_monto) > 0:
    print(sin_monto[['year', 'region', 'buyer', 'amount', 'description']].head(10))

Registros sin monto válido: 53
      year      region                                              buyer  \
6650  2022      GUAYAS  HOSPITAL GENERAL DEL NORTE DE GUAYAQUIL LOS CE...   
6658  2022      GUAYAS  HOSPITAL GENERAL DEL NORTE DE GUAYAQUIL LOS CE...   
6673  2022  CHIMBORAZO       HOSPITAL PROVINCIAL GENERAL DOCENTE RIOBAMBA   
6705  2022      EL ORO                           HOSPITAL GENERAL MACHALA   
6725  2022  CHIMBORAZO       HOSPITAL PROVINCIAL GENERAL DOCENTE RIOBAMBA   
6790  2022       AZUAY   HOSPITAL DE ESPECIALIDADES JOSÉ CARRASCO ARTEAGA   
6848  2022   PICHINCHA               HOSPITAL GENERAL DOCENTE DE CALDERON   
6863  2022        LOJA           HOSPITAL GENERAL MANUEL YGNACIO MONTEROS   
6865  2022   PICHINCHA                    HOSPITAL GENERAL ENRIQUE GARCES   
6870  2022   PICHINCHA                    HOSPITAL SAN FRANCISCO DE QUITO   

      amount                                        description  
6650    0.00  ADQUISICIÓN DE MEDICAMENTOS: GRUPO A10 (I

In [8]:
# Registros sin region
sin_region = df[df['region'].isna() | (df['region'] == '')]
print(f'Registros sin región: {len(sin_region)}')
if len(sin_region) > 0:
    print(sin_region[['year', 'buyer', 'locality', 'amount']].head(10))

Registros sin región: 20
      year                                            buyer locality  amount
225   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   22.50
336   2015          DIRECCION DISTRITAL 16D02-ARAJUNO-SALUD      NaN   30.40
579   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   45.00
581   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   91.20
583   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   45.00
868   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   67.50
870   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   60.80
876   2015  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   67.50
2351  2017  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN   30.40
3746  2019  DIRECCIÓN DISTRITAL 19D01 YACUAMBI-ZAMORA-SALUD      NaN  104.13


### 3.3 Decisiones ETL

Documentar aquí las decisiones tomadas sobre registros problemáticos.

In [9]:
n_antes = len(df)

# Eliminar registros sin monto válido (no aportan al análisis económico)
df = df[df['amount'].notna() & (df['amount'] > 0)]

# Eliminar registros sin región (no se pueden georreferenciar)
df = df[df['region'].notna() & (df['region'] != '')]

n_despues = len(df)
print(f'Registros eliminados: {n_antes - n_despues} ({(n_antes - n_despues)/n_antes*100:.1f}%)')
print(f'Registros finales: {n_despues}')

Registros eliminados: 73 (1.0%)
Registros finales: 7153


### 3.4 Verificar provincias únicas

In [10]:
print(f'Provincias distintas: {df["region"].nunique()}')
print(sorted(df['region'].unique()))

Provincias distintas: 24
['AZUAY', 'BOLIVAR', 'CARCHI', 'CAÑAR', 'CHIMBORAZO', 'COTOPAXI', 'EL ORO', 'ESMERALDAS', 'GALAPAGOS', 'GUAYAS', 'IMBABURA', 'LOJA', 'LOS RIOS', 'MANABI', 'MORONA SANTIAGO', 'NAPO', 'ORELLANA', 'PASTAZA', 'PICHINCHA', 'SANTA ELENA', 'SANTO DOMINGO DE LOS TSACHILAS', 'SUCUMBIOS', 'TUNGURAHUA', 'ZAMORA CHINCHIPE']


## 4. Dataset consolidado limpio

In [11]:
# Columnas relevantes para el análisis
cols = ['id', 'year', 'month', 'date', 'region', 'locality', 'buyer', 'suppliers',
        'amount', 'method', 'internal_type', 'description']
df_clean = df[cols].copy()

# Guardar consolidado completo
out_consolidado = PROCESSED_DIR / 'sercop_insulina_consolidado.csv'
df_clean.to_csv(out_consolidado, index=False)
print(f'Guardado: {out_consolidado} ({len(df_clean)} filas)')
df_clean.head(3)

Guardado: ../processed/sercop_insulina_consolidado.csv (7153 filas)


,id,year,month,date,region,locality,buyer,suppliers,amount,method,internal_type,description
0,1289250,2015,1,2015-01-05 05:00:00+00:00,ORELLANA,FRANCISCO DE ORELLANA,HOSPITAL FRANCISCO DE ORELLANA,LETERAGO DEL ECUADOR S.A,608.00,direct,Catálogo electrónico - Compra directa,Orden de compra para adquirir los siguientes p...
1,1290164,2015,1,2015-01-15 05:00:00+00:00,SANTA ELENA,SALINAS,DIRECCION DISTRITAL 24D02-LA LIBERTAD-SALINAS-...,COMERCIOSA S.A.,450.00,direct,Catálogo electrónico - Compra directa,Orden de compra para adquirir los siguientes p...
2,1291282,2015,1,2015-01-19 05:00:00+00:00,MANABI,PORTOVIEJO,HOSPITAL PROVINCIAL DR VERDI CEVALLOS PORTOVIE...,COMERCIOSA S.A.,315.00,direct,Catálogo electrónico - Compra directa,Orden de compra para adquirir los siguientes p...


## 5. Agregación por provincia y año

Variable clave para el Cruce 1: monto total de compras de insulina por provincia/año.

In [12]:
df_prov_anio = (
    df_clean
    .groupby(['region', 'year'])
    .agg(
        contratos=('id', 'count'),
        monto_total_usd=('amount', 'sum'),
        monto_promedio_usd=('amount', 'mean'),
        compradores_distintos=('buyer', 'nunique')
    )
    .reset_index()
    .rename(columns={'region': 'provincia'})
)
df_prov_anio['monto_total_usd'] = df_prov_anio['monto_total_usd'].round(2)
df_prov_anio['monto_promedio_usd'] = df_prov_anio['monto_promedio_usd'].round(2)

out_prov = PROCESSED_DIR / 'sercop_provincia_anio.csv'
df_prov_anio.to_csv(out_prov, index=False)
print(f'Guardado: {out_prov} ({len(df_prov_anio)} filas)')
df_prov_anio.head(10)

Guardado: ../processed/sercop_provincia_anio.csv (237 filas)


,provincia,year,contratos,monto_total_usd,monto_promedio_usd,compradores_distintos
0,AZUAY,2015,69,"83,376.80","1,208.36",13
1,AZUAY,2016,64,"123,400.99","1,928.14",16
2,AZUAY,2017,43,"149,156.86","3,468.76",16
3,AZUAY,2018,28,"96,738.69","3,454.95",13
4,AZUAY,2019,55,"115,739.16","2,104.35",14
5,AZUAY,2020,49,"94,157.55","1,921.58",16
6,AZUAY,2021,40,"115,987.47","2,899.69",13
7,AZUAY,2022,28,"131,352.90","4,691.18",10
8,AZUAY,2023,12,"160,448.82","13,370.74",5
9,AZUAY,2024,7,"43,357.44","6,193.92",4


## 6. Agregación por provincia (total serie 2015–2024)

Para el cruce con prevalencia de diabetes (ENSANUT es una sola foto, 2018).

In [13]:
df_prov_total = (
    df_clean
    .groupby('region')
    .agg(
        contratos_total=('id', 'count'),
        monto_total_usd=('amount', 'sum'),
        anios_con_datos=('year', 'nunique')
    )
    .reset_index()
    .rename(columns={'region': 'provincia'})
    .sort_values('monto_total_usd', ascending=False)
)
df_prov_total['monto_total_usd'] = df_prov_total['monto_total_usd'].round(2)

out_total = PROCESSED_DIR / 'sercop_provincia_total.csv'
df_prov_total.to_csv(out_total, index=False)
print(f'Guardado: {out_total} ({len(df_prov_total)} filas)')
df_prov_total

Guardado: ../processed/sercop_provincia_total.csv (24 filas)


,provincia,contratos_total,monto_total_usd,anios_con_datos
9,GUAYAS,1273,"11,662,897.36",10
18,PICHINCHA,988,"7,919,798.69",10
13,MANABI,742,"6,289,853.56",10
12,LOS RIOS,259,"1,625,754.92",10
11,LOJA,446,"1,181,503.64",10
6,EL ORO,416,"1,164,700.95",10
0,AZUAY,395,"1,113,716.68",10
20,SANTO DOMINGO DE LOS TSACHILAS,174,"976,236.21",10
4,CHIMBORAZO,205,"835,238.24",10
10,IMBABURA,245,"716,725.16",10


## 7. Serie histórica nacional (para línea de tiempo)

Total de contratos y monto por año — Ecuador completo.

In [14]:
df_nacional = (
    df_clean
    .groupby('year')
    .agg(
        contratos=('id', 'count'),
        monto_total_usd=('amount', 'sum'),
        compradores_distintos=('buyer', 'nunique')
    )
    .reset_index()
)
df_nacional['monto_total_usd'] = df_nacional['monto_total_usd'].round(2)

out_nacional = PROCESSED_DIR / 'sercop_nacional_anio.csv'
df_nacional.to_csv(out_nacional, index=False)
print(f'Guardado: {out_nacional}')
df_nacional

Guardado: ../processed/sercop_nacional_anio.csv


,year,contratos,monto_total_usd,compradores_distintos
0,2015,1112,"2,958,455.96",262
1,2016,1020,"2,766,115.09",296
2,2017,725,"3,169,367.97",266
3,2018,664,"2,676,797.84",276
4,2019,999,"3,392,689.58",299
5,2020,835,"2,188,738.30",281
6,2021,945,"3,776,766.74",260
7,2022,548,"3,577,958.99",218
8,2023,174,"7,782,050.84",88
9,2024,131,"4,797,461.01",76


## 8. Resumen ETL

| Concepto | Valor |
|---|---|
| Registros crudos totales | 7.226 |
| Registros eliminados (sin monto o sin región) | 73 (1,0%) |
| Registros finales | 7.153 |
| Período | 2015–2024 |
| Provincias cubiertas | 24 |
| Outputs generados | sercop_insulina_consolidado.csv, sercop_provincia_anio.csv, sercop_provincia_total.csv, sercop_nacional_anio.csv |

**Nota:** El campo `amount` refleja el monto adjudicado en USD. No representa unidades físicas de insulina — distintos tipos tienen precios muy distintos (glargina vs. NPH vs. regular). Para el análisis usamos monto como proxy de inversión pública, no como volumen de medicamento.

**Sobre los 73 eliminados:** 53 registros con monto nulo o cero (contratos adjudicados sin valor registrado en la plataforma, concentrados en 2022) y 20 sin región asignable (compras de direcciones distritales sin localidad en el sistema). Representan el 1% del total — impacto mínimo en el análisis.